# E3 — Signal-processing ablations

Measure the effect of air referencing, heating/cooling branches, and hysteresis on seven-class identification. All nine representations use the same records, sensor channels, validity masks, feature statistics, and ExtraTrees configuration.

In [ ]:
from pathlib import Path
import hashlib, json, re, sys, os
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

HERE = Path.cwd().resolve()
SRC = next((q for p in (HERE, *HERE.parents) for q in (p, p / "src")
            if (q / "esdata").is_dir()), None)
if SRC is None:
    raise FileNotFoundError("Run this notebook from the repository root or src/")
sys.path.insert(0, str(SRC))
from esdata import EsmellDataset, Chain
from esdata.transforms import (DefaultCSVPreprocessing, SimplifyLabels,
    SelectRChannels, UserLabelCycleSegments, CoolingHeatingSegments,
    CanonicalTempAlign, CanonicalSubtractionFeatures)
from esdata.signal import interpolate_data_by_canonical_temp_grid

DATA_DIR = Path(os.environ.get("ESMELL_DATA_DIR", str(SRC.parent / "data" / "raw")))
OUT = SRC / "outputs" / "E3_repo_exact"
CACHE_DIR = OUT / "cache"
OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVES = {171: "curiosity-5v16.171.0.tar.gz", 172: "curiosity-1v17.172.0.tar.gz"}
GASES = ("acetone", "ammonia", "benzene", "isopropanol", "pentane", "toluene")
CLASSES = ("air",) + GASES
CHANNELS = ("R1", "R2")
GRID = np.linspace(175., 390., 200)
PHASE = {"h": (1, 3, 5), "c": (0, 2, 4)}
SEEDS = (42, 43, 44)
N_TREES = 300
CACHE_VERSION = "E3_repo_exact_v1"
for filename in ARCHIVES.values():
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(DATA_DIR / filename)
print("Results:", OUT)


## 1. Aligned source signals

Load resistance curves and validity masks from `esdata`, applying the same resistance conditioning as the baseline representation.

In [ ]:
def identity(label):
    s = str(label).strip().lower().replace(" ", "").replace("amonia", "ammonia")
    s = s.split("_d", 1)[0]
    if s in ("air", "none"):
        return 0
    if "+" in s:
        raise ValueError(f"Mixture found in the single-gas task: {label}")
    m = re.fullmatch(r"([a-z]+)(?:\[[0-9.]+(?:ppb|ppm)\])?", s)
    if not m or m.group(1) not in GASES:
        raise ValueError(f"Unknown class: {label}")
    return GASES.index(m.group(1)) + 1


def load_card(card):
    archive = DATA_DIR / ARCHIVES[card]
    key = {"version": CACHE_VERSION, "archive": archive.name,
           "bytes": archive.stat().st_size, "mtime_ns": archive.stat().st_mtime_ns,
           "grid": [175, 390, 200], "channels": CHANNELS}
    tag = hashlib.sha256(json.dumps(key, sort_keys=True).encode()).hexdigest()[:16]
    cached = CACHE_DIR / f"aligned_{card}_{tag}.npz"
    keys = ("R", "seg_masks", "weights", "ceiling", "y", "ids", "times")
    if cached.exists():
        with np.load(cached, allow_pickle=False) as z:
            return {k:z[k] for k in keys}

    source = EsmellDataset(root_path=DATA_DIR, subfolders=ARCHIVES[card],
                           csv_repair="strict", archive_cache="disk", preload=True)
    pipe = source >> Chain(
        DefaultCSVPreprocessing(), SimplifyLabels(), SelectRChannels(include=CHANNELS),
        UserLabelCycleSegments(), CoolingHeatingSegments(
            temp_col="T_mean", smooth_window=3, eps=.2),
        CanonicalTempAlign(n_grid=200, temp_range=(175.,390.),
                           normalize=False, conductance=False))
    collected, rejected = [], []
    for sid in source.ids:
        try:
            meta = source.metadata(sid)
            R = np.stack(pipe.features(sid)).astype(np.float32)
            masks = np.stack(pipe.temperature_masks(sid)).astype(bool)
            segments = pipe.segments(sid)
            if R.shape != (6, 2, 200) or masks.shape != (6, 200) or len(segments) != 6:
                raise ValueError("Unexpected segment count or shape")
            if (masks[[1, 3, 5]].all(axis=0)).sum() < 50:
                raise ValueError("Fewer than 50 valid heating temperatures")
            if (masks[[0, 2, 4]].all(axis=0)).sum() < 50:
                raise ValueError("Fewer than 50 valid cooling temperatures")
            ceiling = float(segments[0][list(CHANNELS)].max().max())
            if not (np.isfinite(ceiling) and ceiling >= 1.0):
                raise ValueError(f"Invalid resistance ceiling: {ceiling}")
            t0 = float(segments[0]["Timestamp"].iloc[0])
            times = []
            for segment, mask in zip(segments, masks):
                frame = segment.assign(_seconds=(segment["Timestamp"].to_numpy(dtype=float)-t0)/1000.)
                _, tx, mt, _ = interpolate_data_by_canonical_temp_grid(
                    ["_seconds"], frame, temp_col="T_mean", canonical_temp_grid=GRID)
                if not np.array_equal(mask, mt.astype(bool)):
                    raise ValueError("Time and signal temperature masks differ")
                times.append(tx[:,0])
            times = np.stack(times)
            weights = []
            for p in ("h", "c"):
                a,b,c = PHASE[p]
                valid = masks[[a,b,c]].all(axis=0)
                den = times[c]-times[a]
                if np.any(den[valid]<=0):
                    raise ValueError("Invalid pre-gas-post ordering")
                w = np.divide(times[b]-times[a],den,
                              out=np.full_like(den,.5),where=den>0)
                if np.any((w[valid]<-.01)|(w[valid]>1.01)):
                    raise ValueError("Reference weight outside [0,1]")
                weights.append(np.clip(w,0.,1.))
            collected.append((R,masks,np.stack(weights).astype(np.float32),
                              ceiling,identity(meta["classname"]),str(sid),
                              np.datetime64(pd.Timestamp(meta["datetime"]),"ns")))
        except Exception as e:
            rejected.append((str(sid), str(e)))
    if rejected:
        pd.DataFrame(rejected,columns=["id","reason"]).to_csv(
            OUT/f"qc_rejected_{card}.csv",index=False)
        raise RuntimeError(f"Campaign {card}: {len(rejected)} rejected records; see QC output")
    if not collected:
        raise RuntimeError(f"Campaign {card} is empty")
    data = dict(
        R=np.stack([v[0] for v in collected]),
        seg_masks=np.stack([v[1] for v in collected]),
        weights=np.stack([v[2] for v in collected]),
        ceiling=np.array([v[3] for v in collected],dtype=np.float64),
        y=np.array([v[4] for v in collected],dtype=np.int8),
        ids=np.array([v[5] for v in collected]),
        times=np.array([v[6] for v in collected],dtype="datetime64[ns]"))
    np.savez_compressed(cached, **data)
    print(card, "loaded:",len(data["y"]),"| heating/cooling mask:",
          int(np.median(data["seg_masks"][:,[1,3,5]].all(axis=1).sum(axis=1))),
          int(np.median(data["seg_masks"][:,[0,2,4]].all(axis=1).sum(axis=1))))
    return data

DATA = {c:load_card(c) for c in ARCHIVES}
for card,d in DATA.items():
    mh=d["seg_masks"][:,[1,3,5]].all(axis=1)
    mc=d["seg_masks"][:,[0,2,4]].all(axis=1)
    print(card,"classes:",len(np.unique(d["y"])),
          "| median common mask:",int(np.median((mh&mc).sum(axis=1))))


## 2. Nine input representations

Compare unreferenced, pre-air, two-air, heating, cooling, and hysteresis representations without changing the estimator.

In [ ]:
VARIANTS = ("G_h", "L_h", "pre_h", "fixed_h", "weighted_h",
            "weighted_c", "weighted_hc", "weighted_hh", "fixed_hh")
DISPLAY = {"G_h":"Conductance (heat)", "L_h":"Log conductance (heat)",
    "pre_h":"Pre-air (heat)", "fixed_h":"Fixed bracket (heat)",
    "weighted_h":"Weighted bracket (heat)", "weighted_c":"Weighted bracket (cool)",
    "weighted_hc":"Weighted (heat+cool)",
    "weighted_hh":"Weighted (heat+hyst.)", "fixed_hh":"Fixed (heat+hyst.)"}


def log_conductance(d):
    # Reproduce CanonicalTempAlign(conductance=True): first-air per-sample ceiling
    r = np.clip(d["R"].astype(np.float64),1.0,d["ceiling"][:,None,None,None])
    g = np.clip(1.0/r, 1e-12, 1.0)
    L = np.log(g)
    return np.where(d["seg_masks"][:,:,None,:],L,0.)


def signal(d,variant):
    L = log_conductance(d)
    masks = d["seg_masks"]
    mh = masks[:,[1,3,5]].all(axis=1)
    mc = masks[:,[0,2,4]].all(axis=1)
    mhh = mh & mc
    def phase(p):
        a,b,c=PHASE[p]
        if variant == "G_h": return np.exp(L[:,b])
        if variant == "L_h": return L[:,b]
        if variant == "pre_h": return L[:,b]-L[:,a]
        w = (.5 if variant.startswith("fixed_") else
             d["weights"][:, 0 if p=="h" else 1, None, :])
        return L[:,b]-(1-w)*L[:,a]-w*L[:,c]
    def masked(x,m):
        return np.where(m[:,None,:], x, 0.0)
    h=phase("h")
    if variant=="weighted_c":
        blocks=[masked(phase("c"),mc)]
    elif variant=="weighted_hc":
        blocks=[masked(h,mh),masked(phase("c"),mc)]
    elif variant in ("weighted_hh","fixed_hh"):
        blocks=[masked(h,mh),masked(h-phase("c"),mhh)]
    else:
        blocks=[masked(h,mh)]
    return np.concatenate(blocks,axis=1)


def features(d,variant):
    # Exactly the tree_stats implementation from E2: zero-filled 200-point array.
    x=signal(d,variant)
    if not np.isfinite(x).all():
        raise ValueError(f"Non-finite signal: {variant}")
    blocks=[x.mean(2),x.std(2),x.min(2),x.max(2),
            np.median(x,axis=2),np.quantile(np.abs(x),.9,axis=2)]
    blocks.extend(b.mean(2) for b in np.array_split(x,10,axis=2))
    return np.concatenate(blocks,axis=1).astype(np.float32)

for card,d in DATA.items():
    toy={**d,"weights":np.full_like(d["weights"],.5)}
    np.testing.assert_allclose(signal(toy,"weighted_h"),signal(toy,"fixed_h"),atol=1e-6)
    np.testing.assert_allclose(signal(toy,"weighted_hh"),signal(toy,"fixed_hh"),atol=1e-6)
    hc,hh=signal(d,"weighted_hc"),signal(d,"weighted_hh")
    n=len(CHANNELS)
    mh=d["seg_masks"][:,[1,3,5]].all(axis=1)
    mc=d["seg_masks"][:,[0,2,4]].all(axis=1)
    np.testing.assert_allclose(hh[:,n:],
        np.where((mh&mc)[:,None,:],hc[:,:n]-hc[:,n:],0.),atol=1e-6)
    for v in VARIANTS:
        assert np.isfinite(features(d,v)).all()
    print(card,"9 variants checked, features:",
          {v:features(d,v).shape[1] for v in ("fixed_h","fixed_hh")})


### Canonical-pipeline agreement

Check the reconstructed default features against `CanonicalSubtractionFeatures`, including zero-filled masked regions.

In [ ]:
def verify_exact_pipeline(card):
    source=EsmellDataset(root_path=DATA_DIR,subfolders=ARCHIVES[card],
        csv_repair="strict",archive_cache="disk")
    pipe=source >> Chain(
        DefaultCSVPreprocessing(),SimplifyLabels(),SelectRChannels(include=CHANNELS),
        UserLabelCycleSegments(),CoolingHeatingSegments(
            temp_col="T_mean",smooth_window=3,eps=.2),
        CanonicalTempAlign(n_grid=200,temp_range=(175.,390.),
                           normalize=False,conductance=True),
        CanonicalSubtractionFeatures(features=((1,-3,5),((1,-3,5),(0,-2,4)))))
    d=DATA[card]
    ours=signal(d,"fixed_hh")
    indices=np.unique(np.linspace(0,len(d["ids"])-1,min(12,len(d["ids"])),dtype=int))
    for i in indices:
        sid=str(d["ids"][i]).removesuffix(".csv")
        if sid not in source.ids:
            raise KeyError(f"Sample ID absent from esdata: {sid}")
        expected=np.asarray(pipe.features(sid),dtype=np.float64)
        np.testing.assert_allclose(ours[i],expected,rtol=2e-5,atol=2e-5,
            err_msg=f"{card}, {sid}: baseline differs from esdata")
    print(card,f"matched esdata exactly: {len(indices)} records x 800 values")

for card in DATA:
    verify_exact_pipeline(card)


## 3. Chronological evaluation

Train before 20 October, select using 20 October, and test on 21 October, separately for the two paired campaigns. Three forest seeds measure algorithmic variation.

In [ ]:
def split(d):
    days=d["times"].astype("datetime64[D]")
    tr=np.flatnonzero(days<np.datetime64("2026-10-20"))
    va=np.flatnonzero(days==np.datetime64("2026-10-20"))
    te=np.flatnonzero(days==np.datetime64("2026-10-21"))
    if any(len(ix)==0 for ix in (tr,va,te)):
        raise ValueError(f"Empty split: {[len(ix) for ix in (tr,va,te)]}")
    if any(set(d["y"][ix])!=set(range(7)) for ix in (tr,va,te)):
        raise ValueError("Not all seven classes are present in each split")
    return tr,va,te


def fit_predict(x_train,y_train,x_eval,seed):
    model=ExtraTreesClassifier(n_estimators=N_TREES,min_samples_leaf=1,
        max_features="sqrt",class_weight="balanced",n_jobs=-1,random_state=seed)
    return model.fit(x_train,y_train).predict(x_eval)

SPLITS={c:split(d) for c,d in DATA.items()}
FEATS={(c,v):features(DATA[c],v) for c in DATA for v in VARIANTS}
rows=[]
for card,d in DATA.items():
    tr,va,te=SPLITS[card]
    pd.DataFrame({"class":CLASSES,"train":np.bincount(d["y"][tr],minlength=7),
        "validation":np.bincount(d["y"][va],minlength=7),
        "test":np.bincount(d["y"][te],minlength=7)}).to_csv(
        OUT/f"class_counts_{card}.csv",index=False)
    for v in VARIANTS:
        x=FEATS[card,v]
        for seed in SEEDS:
            pred=fit_predict(x[tr],d["y"][tr],x[va],seed)
            rows.append(dict(card=card,variant=v,seed=seed,
                bacc=balanced_accuracy_score(d["y"][va],pred),
                macro_f1=f1_score(d["y"][va],pred,average="macro",zero_division=0)))
val=pd.DataFrame(rows)
val.to_csv(OUT/"validation_by_seed.csv",index=False)
ranking=val.groupby("variant",sort=False).bacc.mean().sort_values(ascending=False)
SELECTED=str(ranking.index[0])
REFERENCE="fixed_hh"
(OUT/"selection.json").write_text(json.dumps({
    "selected":SELECTED,"criterion":"maximum mean validation BAcc", "ranking":ranking.to_dict(),
    "seeds":SEEDS,"test_previously_inspected":True},ensure_ascii=False,indent=2),encoding="utf-8")
print("Validation BAcc:")
print(ranking.round(4).to_string())
print("Selected:",SELECTED)


In [ ]:
# Final predictions and metrics for all fixed contrasts.
test_rows,predictions=[],[]
for card,d in DATA.items():
    tr,va,te=SPLITS[card]
    fitted=np.r_[tr,va]
    for variant in VARIANTS:
        X=FEATS[card,variant]
        for seed in SEEDS:
            pred=fit_predict(X[fitted],d["y"][fitted],X[te],seed)
            test_rows.append(dict(card=card,variant=variant,seed=seed,
                bacc=balanced_accuracy_score(d["y"][te],pred),
                macro_f1=f1_score(d["y"][te],pred,average="macro",zero_division=0)))
            predictions.extend(dict(card=card,variant=variant,seed=seed,
                sample_id=str(d["ids"][i]),true_class=CLASSES[int(d["y"][i])],
                pred_class=CLASSES[int(p)]) for i,p in zip(te,pred))
test=pd.DataFrame(test_rows)
test.to_csv(OUT/"test_by_seed.csv",index=False)
preds=pd.DataFrame(predictions)
preds.to_csv(OUT/"test_predictions.csv",index=False)

summary=(pd.concat([val.assign(split="val"),test.assign(split="test")])
    .groupby(["variant","card","split"],as_index=False)
    .agg(bacc=("bacc","mean"),sd_seed=("bacc","std"),macro_f1=("macro_f1","mean")))
wide=summary.pivot(index="variant",columns=["card","split"],values="bacc")
wide=wide.reindex(VARIANTS)
wide.columns=[f"{card}_{split}" for card,split in wide.columns]
wide.insert(0,"method",[DISPLAY[v] for v in wide.index])
wide.insert(1,"n_features",[FEATS[171,v].shape[1] for v in wide.index])
wide["mean_val"]=(wide["171_val"]+wide["172_val"])/2
wide["mean_test"]=(wide["171_test"]+wide["172_test"])/2
wide["delta_test_vs_fixed_hh_pp"]=100*(wide["mean_test"]-wide.loc[REFERENCE,"mean_test"])
wide.to_csv(OUT/"table_E3.csv",index_label="variant")
print(wide[["method","n_features","171_val","171_test","172_val","172_test"]].round(3).to_string())

for card in ARCHIVES:
    one=preds[(preds.card==card)&(preds.variant==SELECTED)&(preds.seed==SEEDS[0])]
    cm=confusion_matrix(one.true_class,one.pred_class,labels=CLASSES)
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/f"confusion_{card}.csv")
    pd.DataFrame({"class":CLASSES,"recall":np.divide(np.diag(cm),cm.sum(axis=1),
        out=np.zeros(7,dtype=float),where=cm.sum(axis=1)>0)}).to_csv(
        OUT/f"class_recall_{card}.csv",index=False)

CONTRASTS={
    "Log vs conductance": ("L_h","G_h"),
    "Pre-air referencing": ("pre_h","L_h"),
    "Second air reference": ("fixed_h","pre_h"),
    "Temporal weighting (heat)": ("weighted_h","fixed_h"),
    "Cooling vs heating": ("weighted_c","weighted_h"),
    "Add cooling": ("weighted_hc","weighted_h"),
    "Hysteresis coordinates": ("weighted_hh","weighted_hc"),
    "Temporal weighting (hysteresis)": ("weighted_hh","fixed_hh"),
}
contrasts=pd.DataFrame([dict(comparison=name,alternative=a,control=b,
    **{f"delta_{card}_{sp}_pp":100*(wide.loc[a,f"{card}_{sp}"]-wide.loc[b,f"{card}_{sp}"])
        for card in (171,172) for sp in ("val","test")})
    for name,(a,b) in CONTRASTS.items()])
contrasts.to_csv(OUT/"contrasts_E3.csv",index=False)
print("\nPredefined differences, BAcc percentage points:")
print(contrasts[["comparison","delta_171_test_pp","delta_172_test_pp"]].round(2).to_string(index=False))


## 4. Results

Export validation and test balanced accuracies, class-level errors, and predefined representation contrasts.

In [ ]:
tex=[
    r"\begin{table*}[t]",r"\centering\small",
    r"\caption{E3: fixed-model preprocessing ablations on campaigns 171 and 172.",
    r"Balanced accuracy (\%, mean over three ExtraTrees seeds) on chronological splits.",
    r"All variants use the same legacy resistance clipping and phase masks.}",
    r"\label{tab:e3_ablation}",r"\begin{tabular}{@{}lrrrr@{}}",
    r"\toprule", "Representation & 171 val & 171 test & 172 val & 172 test " + chr(92)*2,
    r"\midrule"]
for v in VARIANTS:
    row=wide.loc[v]
    name=DISPLAY[v].replace("+",r"{+}")
    if v==REFERENCE:
        name=r"\textbf{"+name+"}"
    tex.append(f"{name} & {100*row['171_val']:.1f} & {100*row['171_test']:.1f} "
               f"& {100*row['172_val']:.1f} & {100*row['172_test']:.1f} "+chr(92)*2)
tex += [r"\bottomrule",r"\end{tabular}",
    r"\vspace{2pt}",
    r"\parbox{0.98\textwidth}{\footnotesize The 20 October validation set is",
    r"used for representation selection, and 21 October for descriptive testing.",
    r"For fixed bracket heat+hysteresis, the input tensor reproduces the",
    r"\texttt{CanonicalSubtractionFeatures} pipeline used in E2.}",
    r"\end{table*}"]
(OUT/"E3_table.tex").write_text("\n".join(tex)+"\n",encoding="utf-8")

def delta(a,b,card):
    return 100*(wide.loc[a,f"{card}_test"]-wide.loc[b,f"{card}_test"])

paragraphs=[
    r"\paragraph{Protocol.} We study preprocessing ablations for seven-class gas identity",
    r"classification on two sensor chips (171 and 172) with common channels R1/R2.",
    r"Each variant uses the same resistance clipping, 200-point canonical temperature grid,",
    r"phase-specific validity masks, zero fill outside each output block mask, and",
    r"the same 16 statistical features per sensor-response block as the E2 tree baseline.",
    r"The reference input (fixed-weight heating and hysteresis) numerically matches",
    r"\texttt{CanonicalSubtractionFeatures}. Hyperparameters and chronological splits",
    r"are held fixed; 20 October is used for validation and 21 October for testing.",
    r"All means are over three ExtraTrees random seeds (not independent days).",
    r"Results are listed in Table~\ref{tab:e3_ablation}.",
    "",
    r"\paragraph{Results.} Relative to unreferenced log-conductance, pre-air",
    f"referencing changes test balanced accuracy by {delta('pre_h','L_h',171):+.1f} and {delta('pre_h','L_h',172):+.1f}",
    r"percentage points (171 and 172). Adding the second air reference at equal",
    f"weight changes it by {delta('fixed_h','pre_h',171):+.1f} and {delta('fixed_h','pre_h',172):+.1f} points.",
    r"Compared with fixed weights, time interpolation for the heat+hysteresis",
    f"representation changes it by {delta('weighted_hh','fixed_hh',171):+.1f} and {delta('weighted_hh','fixed_hh',172):+.1f} points.",
    r"Using heat plus hysteresis rather than heat plus cooling changes it",
    f"by {delta('weighted_hh','weighted_hc',171):+.1f} and {delta('weighted_hh','weighted_hc',172):+.1f} points.",
    "",
    r"\paragraph{Interpretation.} The observed effects of air referencing and",
    r"the second air measurement are device- and protocol-dependent.",
    r"Any gain from the heat+hysteresis parameterization is a representation",
    r"effect for the evaluated ExtraTrees model, not new physical information.",
    r"Time-weighted and equal-weight air referencing are compared as alternatives",
    r"without assuming either must outperform the other.",
    r"The test day was inspected in prior exploratory analyses, so these results",
    r"are descriptive and do not establish device-independent optimality."
]
(OUT/"E3_result_draft.tex").write_text("\n".join(paragraphs)+"\n",encoding="utf-8")
(OUT/"E3_protocol.json").write_text(json.dumps({
    "dataset":"171,172 single-gas", "channels":CHANNELS,"n_grid":200,
    "features":"E2 zero-filled 16 statistics per channel/block",
    "resistance_clipping":"first pre-air segment across selected channels",
    "masks":"phase-specific, hysteresis intersection",
    "selected_on_validation":SELECTED,"reference":"fixed_hh",
    "seeds":SEEDS,"tree_params":{"n_estimators":N_TREES,
                              "min_samples_leaf":1,"max_features":"sqrt",
                              "class_weight":"balanced"},
    "splits":{str(c):{s:len(idx) for s,idx in zip(("train","validation","test"),SPLITS[c])}
              for c in DATA},
    "test_previously_inspected":True},ensure_ascii=False,indent=2),encoding="utf-8")
print("LaTeX:",OUT/"E3_table.tex",OUT/"E3_result_draft.tex")
